# Multi-Model TTS Gateway - Synthese Vocale Multi-Modeles

**Module :** 02-Audio-Advanced  
**Niveau :** Avance  
**Technologies :** Kokoro TTS, TADA 3B ML, Qwen3 TTS  
**Duree estimee :** 45 minutes  
**VRAM :** ~12 GB total (2+6+4)  

## Objectifs d'Apprentissage

- [ ] Comprendre l'architecture multi-modele TTS avec gateway
- [ ] Utiliser les 3 modeles via leurs paths differents
- [ ] Comparer la qualite, la vitesse et les caracteristiques de chaque modele
- [ ] Choisir le bon modele selon le cas d'usage
- [ ] Integrer le gateway multi-modele dans une application

## Prerequis

- Gateway TTS multi-modele deploye (port 8196)
- Notebooks 01-5 (Kokoro) et 02-1 (Chatterbox) recommandes
- Comprehension de l'API OpenAI TTS

**Navigation :** [Index](../README.md) | [<< Precedent](02-4-Demucs-Source-Separation.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres TTS
gateway_url = "https://tts-api.myia.io"  # URL du gateway multi-modele
default_voice = "alloy"              # Voix par defaut (OpenAI-compatible)
compare_all_models = True           # Comparer les 3 modeles
benchmark_latency = True            # Benchmarks de latence
save_results = True                 # Sauvegarder les resultats

In [2]:
# Setup environnement et imports
import os
import sys
import json
import time
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
import soundfile as sf
from IPython.display import Audio, display, HTML

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import play_audio, save_audio
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'multi-tts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('multi_tts')

print(f"Multi-Model TTS Gateway - Synthese Vocale")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Gateway : {gateway_url}")
print(f"Sortie : {OUTPUT_DIR}")

Helpers audio importes
Multi-Model TTS Gateway - Synthese Vocale
Date : 2026-04-24 00:33:36
Gateway : https://tts-api.myia.io
Sortie : D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\outputs\audio\multi-tts


L'environnement Python est configure. La cellule suivante charge le fichier `.env` et lit l'URL du gateway TTS, qui peut pointer vers un service local (port 8196) ou le service distant `tts-api.myia.io`.

In [3]:
# Chargement de la configuration .env
from dotenv import load_dotenv

# Remonter jusqu'au dossier GenAI
current_path = Path.cwd()
genai_path = None

while len(current_path.parts) > 1:
    if current_path.name == "GenAI":
        genai_path = current_path
        break
    current_path = current_path.parent

if genai_path:
    env_path = genai_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
    else:
        print("WARNING: .env non trouve dans GenAI")
else:
    print("WARNING: Dossier GenAI non trouve")

# Charger URL du gateway depuis .env si disponible
tts_url = os.getenv('TTS_API_URL', gateway_url)
print(f"TTS Gateway URL: {tts_url}")

.env charge depuis: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\.env
TTS Gateway URL: http://localhost:8191


## Section 1 : Architecture Multi-Model TTS

Le gateway TTS multi-modele expose 3 modeles via des paths differents :

```text
Gateway (Port 8196)
├── /v1/audio/speech         → Kokoro TTS (default)
├── /tada/v1/audio/speech    → TADA 3B ML
└── /qwen/v1/audio/speech    → Qwen3 TTS
```

### Modeles disponibles

| Modele | VRAM | Vitesse | Qualite | Voix | Cas d'usage |
|--------|------|---------|---------|------|------------|
| **Kokoro** | ~2GB | Tres rapide | Bonne | 6 | Prototypage, gros volume |
| **TADA 3B ML** | ~6GB | Rapide | Tres bonne | 1 | Qualite professionnelle |
| **Qwen3 TTS** | ~4GB | Moyen | Excellente | 6+ | Voice cloning, custom |

### Avantages de l'architecture

1. **Backward compatible** : `/v1/audio/speech` utilise Kokoro par defaut
2. **Path-based routing** : Pas de parametre `model` necessaire
3. **Load balancing** : Chaque modele peut etre deploye sur des GPUs differents
4. **Lazy loading** : Les modeles se chargent a la demande
5. **OpenAI-compatible** : Meme API que OpenAI TTS

In [4]:
# Verification du gateway et des modeles
print("VERIFICATION DU GATEWAY")
print("=" * 50)

try:
    # Health check
    response = requests.get(f"{tts_url}/health", timeout=10)
    health = response.json()
    print(f"Status: {health['status']}")
    print(f"Modeles disponibles: {', '.join(health['models'])}")
    
    # Lister les modeles
    response = requests.get(f"{tts_url}/v1/models", timeout=10)
    models_info = response.json()
    print(f"\nModeles enregistres:")
    for model in models_info['data']:
        print(f"  - {model['id']}: {model['name']}")
        print(f"    Path: {model['path']}")
        print(f"    Description: {model['description']}")
    
    # Lister les voix
    response = requests.get(f"{tts_url}/v1/voices", timeout=10)
    voices_info = response.json()
    print(f"\nVoix disponibles:")
    for voice in voices_info['voices']:
        models = ', '.join(voice['models'])
        print(f"  - {voice['id']}: {voice['name']} (modeles: {models})")
    
    print("\nGateway operationnel!")
    
except requests.exceptions.ConnectionError:
    print("ERREUR: Gateway non accessible")
    print(f"Verifiez que le service tourne sur {tts_url}")
except Exception as e:
    print(f"ERREUR: {str(e)[:100]}")

VERIFICATION DU GATEWAY
Status: healthy
ERREUR: 'models'


## Section 2 : Generation avec les 3 Modeles

Comparaison directe des 3 modeles sur le meme texte.

In [5]:
# Comparaison des 3 modeles TTS
print("COMPARAISON DES 3 MODELES TTS")
print("=" * 50)

# Texte de test - court pour Qwen3 (lent ~1s/mot), normal pour les autres
sample_text = (
    "L'intelligence artificielle transforme la facon dont nous "
    "interagissons avec la technologie. La synthese vocale est "
    "devenue si naturelle qu'il est parfois difficile de distinguer "
    "une voix humaine d'une voix artificielle."
)

# Texte court pour Qwen3 (modele lent, ~1min par phrase)
sample_text_short = (
    "La synthese vocale est devenue si naturelle "
    "qu'il est difficile de distinguer une voix artificielle."
)

# Configuration des modeles
models_config = {
    "kokoro": {
        "url": f"{tts_url}/v1/audio/speech",
        "name": "Kokoro",
        "description": "Modele leger et rapide",
        "text": sample_text,
        "timeout": 60
    },
    "tada": {
        "url": f"{tts_url}/tada/v1/audio/speech",
        "name": "TADA 3B ML",
        "description": "Modele avance de HumeAI",
        "text": sample_text,
        "timeout": 60
    },
    "qwen": {
        "url": f"{tts_url}/qwen/v1/audio/speech",
        "name": "Qwen3 TTS",
        "description": "Modele haute qualite de Qwen (lent, ~1min/phrase)",
        "text": sample_text_short,
        "timeout": 180
    }
}

results = {}

for model_key, config in models_config.items():
    print(f"\n--- {config['name']} ---")
    print(f"Description: {config['description']}")
    
    try:
        start_time = time.time()
        
        # Pas de parametre "model" - le routage se fait par le path
        # vLLM (Qwen3) rejette les noms de modeles inconnus
        response = requests.post(
            config['url'],
            json={
                "input": config['text'],
                "voice": default_voice
            },
            timeout=config['timeout']
        )
        
        if response.status_code == 200:
            gen_time = time.time() - start_time
            audio_bytes = response.content
            
            # Verifier que c'est bien de l'audio (pas une erreur JSON)
            content_type = response.headers.get('content-type', '')
            if 'json' in content_type:
                print(f"  ERREUR: {response.json()}")
                continue
            
            # Charger pour connaitre la duree
            import io
            audio_data, sample_rate = sf.read(io.BytesIO(audio_bytes))
            duration = len(audio_data) / sample_rate
            
            results[model_key] = {
                "audio": audio_bytes,
                "duration": duration,
                "gen_time": gen_time,
                "sample_rate": sample_rate,
                "size_kb": len(audio_bytes) / 1024,
                "text_len": len(config['text'])
            }
            
            print(f"  Texte: {len(config['text'])} chars")
            print(f"  Duree audio: {duration:.1f}s")
            print(f"  Temps de generation: {gen_time:.2f}s")
            print(f"  Ratio temps reel: {duration/gen_time:.1f}x")
            print(f"  Taille: {len(audio_bytes)/1024:.1f} KB")
            
            # Ecouter
            display(Audio(data=audio_bytes, rate=sample_rate))
            
            # Sauvegarder
            if save_results:
                filepath = OUTPUT_DIR / f"{model_key}_comparison.wav"
                with open(filepath, 'wb') as f:
                    f.write(audio_bytes)
                print(f"  Sauvegarde: {filepath.name}")
        else:
            print(f"  ERREUR: Status {response.status_code}")
            print(f"  {response.text[:200]}")
            
    except requests.exceptions.Timeout:
        gen_time = time.time() - start_time
        print(f"  TIMEOUT apres {gen_time:.0f}s (limite: {config['timeout']}s)")
        print(f"  Qwen3 est tres lent (~1min/phrase). Essayez un texte plus court.")
    except Exception as e:
        print(f"  ERREUR: {str(e)[:100]}")

COMPARAISON DES 3 MODELES TTS

--- Kokoro ---
Description: Modele leger et rapide
  ERREUR: Status 401
  {"error": {"message": "Missing or invalid Authorization header", "type": "authentication_error"}}

--- TADA 3B ML ---
Description: Modele avance de HumeAI
  ERREUR: Status 401
  {"error": {"message": "Missing or invalid Authorization header", "type": "authentication_error"}}

--- Qwen3 TTS ---
Description: Modele haute qualite de Qwen (lent, ~1min/phrase)
  ERREUR: Status 401
  {"error": {"message": "Missing or invalid Authorization header", "type": "authentication_error"}}


Les trois modeles ont ete interroges. La cellule suivante construit un tableau comparatif synthetisant les mesures de duree, de temps de generation et de ratio temps reel pour faciliter la prise de decision.

In [6]:
# Tableau comparatif
if results:
    print("\n" + "=" * 80)
    print("TABLEAU COMPARATIF")
    print("=" * 80)
    
    print(f"{'Modele':<15} {'Chars':<8} {'Duree (s)':<12} {'Temps (s)':<12} {'Ratio':<10} {'KB':<10}")
    print("-" * 67)
    
    for model_key, data in results.items():
        ratio = data['duration'] / data['gen_time'] if data['gen_time'] > 0 else 0
        print(f"{models_config[model_key]['name']:<15} "
              f"{data['text_len']:<8} "
              f"{data['duration']:<12.1f} "
              f"{data['gen_time']:<12.2f} "
              f"{ratio:<10.1f} "
              f"{data['size_kb']:<10.1f}")
    
    # Recommandation
    if len(results) > 1:
        fastest = max(results.items(), key=lambda x: x[1]['duration'] / x[1]['gen_time'] if x[1]['gen_time'] > 0 else 0)
        print(f"\nPlus rapide (ratio temps reel): {models_config[fastest[0]]['name']}")
    
    print(f"\nNote: Qwen3 utilise un texte plus court car il est ~10x plus lent que Kokoro")
    print(f"      TADA offre un bon compromis qualite/vitesse")

### Exercice 1 : Comparaison personnalisee avec parametres

L'exemple ci-dessus compare les 3 modeles avec un texte court et des parametres par defaut. **Votre mission** : testez les modeles avec des textes plus longs et des parametres differents pour comprendre l'impact sur la qualite et le temps de generation.

**Indices** :
- Utilisez un texte de 100+ mots (extrait de Wikipedia ou autre)
- Testez avec `speed` differents (0.5, 1.0, 1.5, 2.0) pour Kokoro
- Mesurez le temps de generation pour chaque configuration
- Observez comment la longueur du texte affecte le ratio real-time


In [7]:
# --- Exercice 1 : Comparaison personnalisee ---

def compare_tts_long_text(text, models=None, speed_values=None):
    """
    Compare les modeles TTS avec un texte long et des vitesses differentes.
    Retourne un DataFrame avec les resultats.
    """
    # TODO: Pour chaque modele, generer l'audio avec le texte long
    # TODO: Pour Kokoro, tester differentes valeurs de speed
    # TODO: Mesurer le temps de generation et la duree audio
    # TODO: Calculer le ratio real-time pour chaque configuration
    pass

# Test
# long_text = "Votre texte de 100+ mots ici..."
# df = compare_tts_long_text(long_text)
# print(df.to_string())


## Section 3 : Guide de Selection de Modele

### Arbre de decision

```text
Besoin TTS
    │
    ├─ Prototypage / Tests → Kokoro (rapide, gratuit)
    ├─ Production standard → TADA 3B ML (qualite, vitesse)
    ├─ Haute qualite → Qwen3 TTS (meilleure naturalite)
    ├─ Gros volume → Kokoro (pas de cout per-token)
    └─ Voice cloning → Qwen3 TTS (custom voices)
```

### Cas d'usage par modele

| Cas d'usage | Modele recommande | Pourquoi |
|------------|------------------|---------|
| Prototypage rapide | Kokoro | Instantane, pas d'attente |
| Audiobooks | TADA 3B ML | Voix claire, professionnelle |
| Chatbots vocaux | Kokoro | Reponse rapide, faible latence |
| Voice cloning | Qwen3 TTS | Support voix customisees |
| Applications mobiles | Kokoro | Modele leger, fonctionne sur CPU |
| Contenu marketing | Qwen3 TTS | Meilleure qualite perceptuelle |

## Section 4 : Integration dans une Application

### Exemple de code Python

```python
import requests

class MultiModelTTSClient:
    """Client TTS multi-modele."""
    
    def __init__(self, gateway_url="https://tts-api.myia.io"):
        self.gateway_url = gateway_url
        self.models = {
            "kokoro": "/v1/audio/speech",
            "tada": "/tada/v1/audio/speech",
            "qwen": "/qwen/v1/audio/speech"
        }
    
    def generate(self, text: str, model: str = "kokoro", voice: str = "alloy"):
        """Generer de la parole avec le modele specifie."""
        url = self.gateway_url + self.models[model]
        
        response = requests.post(
            url,
            json={"input": text, "voice": voice},
            timeout=60
        )
        
        if response.status_code == 200:
            return response.content
        else:
            raise Exception(f"TTS error: {response.status_code}")

# Utilisation
client = MultiModelTTSClient()
audio = client.generate("Bonjour!", model="tada", voice="alloy")
```

In [8]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 50)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Gateway : {tts_url}")
print(f"Modeles testes : {len(results)}")

if save_results:
    saved = list(OUTPUT_DIR.glob('*'))
    total_size = sum(f.stat().st_size for f in saved) / (1024**2)
    print(f"Fichiers sauvegardes : {len(saved)} ({total_size:.1f} MB)")

print(f"\nPROCHAINES ETAPES")
print(f"1. Integrer le TTS dans une application FastAPI (03-1)")
print(f"2. Construire un pipeline vocal complet (03-2)")
print(f"3. Explorer le voice cloning avec XTTS (02-2)")
print(f"4. Comparer tous les modeles TTS et STT (03-1)")

print(f"\nNotebook Multi-Model TTS termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-04-24 00:33:36
Gateway : http://localhost:8191
Modeles testes : 0


Fichiers sauvegardes : 0 (0.0 MB)

PROCHAINES ETAPES
1. Integrer le TTS dans une application FastAPI (03-1)
2. Construire un pipeline vocal complet (03-2)
3. Explorer le voice cloning avec XTTS (02-2)
4. Comparer tous les modeles TTS et STT (03-1)

Notebook Multi-Model TTS termine - 00:33:36


### Exercice 2 : Routeur TTS intelligent

La Section 3 presente un arbre de decision pour choisir le modele adapte. **Votre mission** : implementez une fonction `smart_tts_router()` qui selectionne automatiquement le meilleur modele et genere l'audio.

**Indices** :
- Criteres de selection : longueur du texte, langue, besoin de clonage vocal, contrainte de temps
- Texte court (< 50 mots) + rapidite -> Kokoro
- Texte long + qualite -> Qwen3 TTS
- Clonage vocal -> Qwen3 TTS (parametre `voice`)
- Ajoutez un fallback si un modele est indisponible


In [9]:
# --- Exercice 2 : Routeur TTS intelligent ---

def smart_tts_router(text, voice_clone=None, max_wait_seconds=30):
    """
    Selectionne automatiquement le meilleur modele TTS et genere l'audio.
    Criteres : longueur, clonage vocal, contrainte de temps.
    Retourne (audio_bytes, model_used, generation_time).
    """
    # TODO: Verifier la disponibilite de chaque modele
    # TODO: Estimer le temps de generation selon la longueur du texte
    # TODO: Si voice_clone demande -> Qwen3 TTS
    # TODO: Si texte court + rapide -> Kokoro
    # TODO: Si texte long + qualite -> Qwen3 ou TADA
    # TODO: Implementer un fallback si le modele choisi echoue
    pass

# Test
# audio, model, t = smart_tts_router("Bonjour le monde")
# print(f"Modele: {model}, Temps: {t:.1f}s")
